# Modul 12 · Experimente und A/B-Tests
**Data Science 1** · linkit courses

Wie zeige ich, dass eine Maßnahme wirkt? Das Skript wertet die Rauheit im Werkversuch an M-04 und die Bestellquote im Ersatzteilportal aus. Du siehst dir die Regression zur Mitte bei den besten Maschinen an, ziehst eine Zuteilung selbst und prüfst, ob die Kovariate vor der Behandlung feststand. Du rechnest die Schleifzeit gepaart, testest raue Lose mit Vorzeichentausch und nimmst die Reihenfolge als zweiten Faktor auf. Im Portal planst du die Teststärke, prüfst die Zuteilung, rechnest einen Bootstrap über Konten, CUPED mit dem Vorjahr und eine A/A-Simulation zum Peeking. Am Ende prüfst du eine begründete Teilgruppe und schreibst den Entscheidungsbericht.

**Lernziele**
1. Erklären, warum Vorher-nachher-Vergleiche Wirkungen vortäuschen können
2. Versuche randomisieren und die Einheit der Zuteilung festlegen
3. Unterschiede mit Intervall, Permutationstest und t-Test auswerten
4. Blöcke und Kovariaten nutzen, um genauer zu messen
5. Stichprobenumfang und Teststärke vor dem Versuch planen
6. A/B-Tests auf Zuteilungsfehler, Peeking und Teilgruppen prüfen
7. Mit Haupt- und Nebenkennzahlen eine begründete Entscheidung treffen

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus. Der erste Import lädt im Browser SciPy und statsmodels, das dauert einmalig etwa 10 Sekunden.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis zehn Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Die TODOs bauen nur selten aufeinander auf. Wo ein späteres TODO ein früheres Ergebnis braucht, springt eine Vorgabe-Zelle ein, solange das frühere offen ist. Du kannst die TODOs also auch einzeln bearbeiten.
- Mindestziel sind TODO 1 bis 13. Die beiden Bonusaufgaben vertiefen.
- Aller Zufall hängt an `SEED = 12` in der Zelle nach den Importen. Jede Zelle mit Zufall legt ihren Generator mit `np.random.default_rng(SEED)` neu an und liefert so bei jedem Lauf dasselbe. Die Selbsttests lassen so viel Spielraum, dass jeder richtige Weg auch mit einem anderen Seed besteht.
- Gib Ergebnisse mit `float(...)`, `round(...)` oder der Hilfsfunktion `zeige` aus, dann steht im Browser und in Colab dasselbe da. Jede Grafikzelle endet mit `plt.show()`.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["schleifversuch.csv", "haerterei.csv", "portal_konten.csv", "portal_sitzungen.csv", "stoerungen.csv", "maschinen.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Vorher-nachher ist kein Nachweis
*Lerneinheit 1 · Skript Abschnitt 1*

Ein Vorher-nachher-Vergleich nimmt den Wert vor einer Maßnahme als Ersatz für das Ergebnis ohne Maßnahme. Das täuscht, sobald sich sonst etwas ändert: Zeit und Saison, die Auswahl der Einheiten oder die **Regression zur Mitte**. Wer Einheiten wegen eines extremen Werts auswählt, misst beim nächsten Mal meist einen Wert näher am Mittel, auch ganz ohne Maßnahme. Im Skript stehen die drei Maschinen mit dem meisten Stillstand im Folgemonat deutlich kürzer still.

Die nächste Zelle lädt die Störungen wie im Skript und baut `monat`, den Stillstand in Minuten je Monat (Zeilen) und Maschine (Spalten). Die Hilfsfunktion `zeige` druckt Zahlen mit 6 gültigen Stellen, Tabellen mit ihrer Größe und lange Arrays mit Länge und Summe. Die Zelle danach legt den Seed für alles fest, was in diesem Notebook zufällig ist.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

stoerungen = pd.read_csv("daten/stoerungen.csv", parse_dates=["datum"])
maschinen = pd.read_csv("daten/maschinen.csv")
monat = (stoerungen.assign(monat=stoerungen["datum"].dt.to_period("M"))
         .pivot_table(index="monat", columns="maschine", values="stillstand_min", aggfunc="sum", fill_value=0)
         .reindex(columns=maschinen["maschine"], fill_value=0))
print(monat.shape, round(monat.to_numpy().mean(), 1))


def zeige(*werte):
    """Druckt Zahlen mit 6 gültigen Stellen, damit im Browser und in Colab dieselben Ziffern stehen.
    Zeitstempel erscheinen als Datum, Tabellen mit ihrer Größe, lange Arrays mit Länge und Summe. None bleibt None."""
    def text(w):
        if w is None or isinstance(w, (str, bool, np.bool_)):
            return str(w)
        if isinstance(w, pd.Timestamp):
            return w.strftime("%d.%m.%Y")
        if isinstance(w, dict):
            return "{" + ", ".join(f"{text(k)}: {text(v)}" for k, v in w.items()) + "}"
        if isinstance(w, tuple):
            return "(" + ", ".join(text(v) for v in w) + ")"
        if isinstance(w, pd.DataFrame):
            return f"Tabelle {w.shape[0]} × {w.shape[1]}"
        if np.ndim(w) == 0:
            return f"{float(w):.6g}"
        w = list(np.ravel(np.asarray(w, dtype=object)))
        if len(w) > 12:
            try:
                return f"[{len(w)} Werte, Summe {float(np.nansum(np.asarray(w, dtype=float))):.6g}]"
            except (TypeError, ValueError):
                return f"[{len(w)} Werte]"
        return "[" + " ".join(text(v) for v in w) + "]"
    print(*[text(w) for w in werte])

In [ ]:
# Ein Seed für alles: Jede Zelle mit Zufall legt ihren Generator mit np.random.default_rng(SEED) neu an.
SEED = 12

### TODO 1 · Regression zur Mitte bei den Besten
Und in die andere Richtung? Angenommen, die Instandhaltung lobt jeden Monat die drei Maschinen mit dem wenigsten Stillstand.
1. `vorher` und `nachher`: zwei Arrays mit je 23 Werten. Für jeden Monat mit Folgemonat wählst du mit `nsmallest(3)` die drei Maschinen mit dem wenigsten Stillstand und nimmst ihren mittleren Stillstand im Monat der Auswahl (`vorher`) und für **dieselben drei Maschinen** im Folgemonat (`nachher`).
2. `schlechter`: in wie vielen der 23 Monate `nachher` über `vorher` liegt.

In [ ]:
# TODO 1
vorher = None
nachher = None
schlechter = None

zeige(None if vorher is None else np.mean(vorher), None if nachher is None else np.mean(nachher), schlechter)

In [ ]:
check("TODO 1 Auswahl und Folgemonat", lambda: len(vorher) == 23 and len(nachher) == 23 and abs(np.mean(vorher) - 599.7) < 0.05 and abs(np.mean(nachher) - 875.2) < 0.05)
check("TODO 1 Monate mit Verschlechterung", lambda: schlechter == 19)

Die drei Besten eines Monats stehen im Monat der Auswahl im Mittel 599,7 Minuten still, im Folgemonat 875,2 Minuten, gut 275 Minuten mehr. In 19 von 23 Monaten verschlechtern sie sich. Eine Maßnahme gab es nicht. Wer wegen eines guten Werts ausgewählt wird, hatte oft auch Glück, und das Glück wiederholt sich im Mittel nicht. Wer die Besten lobt und danach misst, sieht einen Rückfall; wer die Schlechtesten ermahnt, sieht eine Besserung. Beides zeigt nur, wie ausgewählt wurde.

## 2 · Randomisieren und planen
*Lerneinheit 2 · Skript Abschnitt 2*

Entscheidet ein Zufallsgenerator, wer die Maßnahme bekommt, hängt die Zuteilung mit nichts zusammen, auch nicht mit Merkmalen, die niemand gemessen hat. Im Mittel sind die Gruppen dann gleich, im einzelnen Versuch bleibt eine kleine Ungleichheit. Wie groß sie ist, zeigt für Merkmale von vor dem Versuch die **standardisierte Mittelwertdifferenz**

$$\text{SMD} = \frac{\bar x_B - \bar x_A}{\sqrt{(s_A^2 + s_B^2)/2}}.$$

Sie misst den Unterschied in Standardabweichungen und hängt nicht von der Zahl der Einheiten ab. Beträge unter 0,1 gelten als unauffällig. Eine Kovariate muss außerdem **vor der Behandlung** feststehen, sonst kann die Behandlung sie verändern.

Die nächste Zelle lädt den Werkversuch mit der Schicht als Block wie im Skript, die Härterei mit dem Härtetag als Datum und die Konten des Portaltests.

In [ ]:
versuch = pd.read_csv("daten/schleifversuch.csv", parse_dates=["datum"])
versuch["block"] = versuch["datum"].dt.strftime("%Y-%m-%d") + " " + versuch["schicht"]
haerterei = pd.read_csv("daten/haerterei.csv", parse_dates=["datum"])
konten = pd.read_csv("daten/portal_konten.csv")
print(versuch.shape, versuch["block"].nunique(), haerterei.shape, konten.shape, konten["gruppe"].value_counts().to_dict())

### TODO 2 · Zuteilung reproduzierbar ziehen und prüfen
Wie gut gleicht eine Zuteilung aus? Vergleiche die echte Zuteilung des Portals mit einer, die du selbst ziehst.
1. `neu`: eine neue Zuteilung der 2.548 Konten als Array mit den Werten „A“ und „B“. Lege `rng = np.random.default_rng(SEED)` an; ein Konto kommt nach B, wenn seine Zahl aus `rng.random(len(konten))` unter 0,5 liegt.
2. `ausgleich`: eine Tabelle mit den Zeilen `umsatz_vorjahr_eur` und `umsatz_vorperiode_eur` und den Spalten `echt` (Gruppen aus `konten["gruppe"]`) und `neu`. Darin steht die standardisierte Differenz B minus A, mit den Varianzen von pandas (`ddof=1`).

In [ ]:
# TODO 2
rng = np.random.default_rng(SEED)
neu = None
ausgleich = None

zeige(None if neu is None else int((np.asarray(neu) == "B").sum()))
print(None if ausgleich is None else ausgleich.round(3))

In [ ]:
def _smd(x, g):
    x, g = np.asarray(x, float), np.asarray(g)
    return (x[g == "B"].mean() - x[g == "A"].mean()) / np.sqrt((x[g == "A"].var(ddof=1) + x[g == "B"].var(ddof=1)) / 2)

check("TODO 2 Zuteilung mit dem Seed", lambda: len(neu) == len(konten) and set(np.asarray(neu)) == {"A", "B"}
      and np.array_equal(np.asarray(neu) == "B", np.random.default_rng(SEED).random(len(konten)) < 0.5))
check("TODO 2 standardisierte Differenzen", lambda: list(ausgleich.columns) == ["echt", "neu"] and list(ausgleich.index) == ["umsatz_vorjahr_eur", "umsatz_vorperiode_eur"]
      and abs(ausgleich.loc["umsatz_vorjahr_eur", "echt"] + 0.062) < 0.0005 and abs(ausgleich.loc["umsatz_vorperiode_eur", "echt"] - 0.024) < 0.0005
      and all(abs(ausgleich.loc[s, "neu"] - _smd(konten[s], neu)) < 1e-9 for s in ausgleich.index) and bool((ausgleich.abs() < 0.2).all().all()))

Die neue Zuteilung legt 1.282 Konten nach B. Beim Vorjahresumsatz liegt die standardisierte Differenz echt bei −0,062 und neu bei 0,052, beim Umsatz der Vorperiode echt bei 0,024 und neu bei −0,021. Alle Beträge liegen unter 0,1: Beide Zuteilungen gleichen gut aus, eine kleine Ungleichheit bleibt in jeder, mal zugunsten von A, mal von B. Mit demselben Seed entsteht bei jedem Lauf dieselbe Zuteilung, so lässt sie sich prüfen und wiederholen.

### TODO 3 · Steht die Kovariate vor der Behandlung fest?
Im Skript nimmt die Auswertung die Härte der Charge als Kovariate. Das ist nur erlaubt, wenn die Härte vor dem Schleifen feststand und die Scheibe sie nicht beeinflussen kann. Prüfe das.
1. `lose`: `versuch` mit Härtetag (als Spalte `datum_haerte`), `haerte_hrc` und `in_toleranz` aus `haerterei`, angehängt über `charge_id` mit `merge(..., how="left", validate="many_to_one")`. Die Prüfung meldet einen Fehler, falls eine Charge doppelt in der Härterei steht.
2. `tage_bis`: wie viele Lose wie viele Tage nach dem Härten geschliffen wurden, als Series mit den Tagen als Index (`value_counts().sort_index()`).
3. `haerte_mittel`: die mittlere Härte je Scheibe als Series. `dh`: die Härtedifferenz B minus A je Schicht als Series mit `block` als Index. `r_haerte`: die Korrelation von `dh` mit der Differenz der Rauheit B minus A je Schicht.

In [ ]:
# TODO 3
lose = None
tage_bis = None
haerte_mittel = None
dh = None
r_haerte = None

zeige(None if lose is None else (len(lose), lose["charge_id"].nunique(), bool(lose["in_toleranz"].eq(1).all())), None if tage_bis is None else tage_bis.to_dict(),
      None if haerte_mittel is None else haerte_mittel.to_dict(), None if dh is None else (dh.mean(), int((dh > 0).sum())), r_haerte)

In [ ]:
check("TODO 3 Härte je Los", lambda: len(lose) == 120 and lose["charge_id"].nunique() == 103 and bool(lose["in_toleranz"].eq(1).all())
      and pd.api.types.is_datetime64_any_dtype(lose["datum_haerte"]) and tage_bis.to_dict() == {1: 96, 3: 24})
check("TODO 3 Ungleichgewicht je Schicht", lambda: abs(haerte_mittel["A"] - 55.18) < 0.005 and abs(haerte_mittel["B"] - 55.33) < 0.005 and len(dh) == 60
      and abs(dh.mean() - 0.148) < 0.0005 and int((dh > 0).sum()) == 32 and abs(r_haerte - 0.707) < 0.0005)

Die 120 Lose stammen aus 103 Chargen, alle in Toleranz. 96 Lose wurden einen Tag nach dem Härten geschliffen, 24 drei Tage danach, über das Wochenende. Die Härte stand also fest, bevor die Scheibe ins Spiel kam, und die Scheibe kann sie nicht verändern. Mit B lag die Härte im Mittel bei 55,33 HRC, mit A bei 55,18. In 32 der 60 Schichten war das Los mit B härter, im Mittel um 0,148 HRC je Schicht. Die Härtedifferenz hängt mit der Rauheitsdifferenz mit 0,707 zusammen. Wer die Härte nicht berücksichtigt, rechnet diesen Teil der Scheibe zu.

## 3 · Den Werkversuch auswerten
*Lerneinheit 3 · Skript Abschnitt 3*

Im Block wurde innerhalb jeder Schicht gelost. Die Auswertung folgt diesem Plan: Sie vergleicht A und B innerhalb der Schicht, über die Differenzen $d_i = y_{B,i} - y_{A,i}$. Der gepaarte t-Test hat den Standardfehler $s_d/\sqrt{n}$, der Welch-Test über alle Lose $\sqrt{s_A^2/n_A + s_B^2/n_B}$. Der Permutationstest im Block tauscht die Vorzeichen der Differenzen, weil unter der Nullhypothese jede Schicht ihre zwei Werte auch andersherum hätte bekommen können.

Die nächste Zelle bildet `paare` und `d` für die Rauheit wie im Skript und zeigt die Rauheit von A und B je Schicht.

In [ ]:
paare = versuch.pivot(index="block", columns="scheibe", values="ra_um")
d = paare["B"] - paare["A"]
gepaart = stats.ttest_rel(paare["B"], paare["A"])
print(paare.shape, round(d.mean(), 4), round(d.std() / np.sqrt(len(d)), 4), round(float(gepaart.pvalue), 3), int((d < 0).sum()))

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(paare["A"], paare["B"], s=18, color="tab:blue")
grenzen = [paare.min().min() - 0.01, paare.max().max() + 0.01]
ax.plot(grenzen, grenzen, color="gray", lw=1, ls="--")
ax.set_xlabel("Rauheit mit A in µm")
ax.set_ylabel("Rauheit mit B in µm")
ax.set_title("Je Schicht ein Punkt: unter der Linie ist B glatter")
plt.show()

### TODO 4 · Schleifzeit ungepaart und gepaart
Die Regel im Plan erlaubt höchstens 3 % mehr Schleifzeit. Im Skript steht nur das Ergebnis, hier rechnest du es nach.
1. `paare_zeit`: die Schleifzeit je Schicht (Zeilen `block`) und Scheibe (Spalten), mit `pivot` wie `paare`. `diff_zeit`: das Mittel der Differenzen B minus A.
2. `ungepaart_zeit`: ein Wörterbuch mit dem Standardfehler `"se"` und `"t"` aus dem Welch-Test über alle Lose.
3. `gepaart_zeit`: ein Wörterbuch mit `"se"`, `"t"`, `"unten"` und `"oben"` (95-%-Intervall) aus dem gepaarten t-Test. `rel_zeit`: Differenz und Intervallgrenzen in Prozent der mittleren Schleifzeit mit A, als Wörterbuch mit `"diff"`, `"unten"` und `"oben"`. `rho_zeit`: die Korrelation von A und B innerhalb der Schicht.

In [ ]:
# TODO 4
paare_zeit = None
diff_zeit = None
ungepaart_zeit = None
gepaart_zeit = None
rel_zeit = None
rho_zeit = None

zeige(diff_zeit, ungepaart_zeit, gepaart_zeit, rel_zeit, rho_zeit)

In [ ]:
check("TODO 4 ungepaart", lambda: paare_zeit.shape == (60, 2) and abs(diff_zeit - 1.042) < 0.0005 and abs(ungepaart_zeit["se"] - 0.1482) < 0.00005
      and abs(ungepaart_zeit["t"] - 7.03) < 0.005)
check("TODO 4 gepaart", lambda: abs(gepaart_zeit["se"] - 0.0969) < 0.00005 and abs(gepaart_zeit["t"] - 10.75) < 0.005 and abs(gepaart_zeit["unten"] - 0.848) < 0.0005
      and abs(gepaart_zeit["oben"] - 1.236) < 0.0005 and abs(rel_zeit["diff"] - 2.55) < 0.005 and abs(rel_zeit["unten"] - 2.07) < 0.005
      and abs(rel_zeit["oben"] - 3.02) < 0.005 and abs(rho_zeit - 0.581) < 0.0005)

B schleift im Mittel 1,042 s länger je Welle, 2,55 % mehr als A. Ungepaart ist der Standardfehler 0,1482 s und $t = 7{,}03$, gepaart 0,0969 s und $t = 10{,}75$. Innerhalb der Schicht hängen die Schleifzeiten von A und B mit 0,581 zusammen, schwächer als die Rauheit, aber genug, um den Standardfehler um gut ein Drittel zu senken. Das 95-%-Intervall reicht von 0,848 bis 1,236 s, also von 2,07 bis 3,02 % mehr. Die Schätzung liegt unter der Grenze von 3 %, die obere Grenze des Intervalls genau daran.

### TODO 5 · Permutationstest im Block für raue Lose
Die Qualitätssicherung zählt oft nur, ob ein Los die Grenze von 0,45 µm überschreitet. Prüfe den Anteil rauer Lose so, wie gelost wurde.
1. `d_rau`: je Schicht 1 für ein raues Los mit B minus 1 für ein raues Los mit A, als Series mit `block` als Index. Die Werte sind −1, 0 oder 1.
2. `p_rau`: der p-Wert des Permutationstests mit Vorzeichentausch wie im Skript. Lege `rng = np.random.default_rng(SEED)` an, ziehe mit `rng.choice([-1, 1], size=(10_000, 60))` alle Vorzeichenmuster auf einmal und rechne $p = (b+1)/(R+1)$.
3. `p_rau_exakt`: der exakte Wert. Schichten mit $d_i = 0$ ändern sich beim Vorzeichentausch nicht. Von den übrigen ist unter der Nullhypothese jede mit Wahrscheinlichkeit 1/2 positiv, also passt `stats.binomtest` mit der Zahl der positiven unter allen von 0 verschiedenen Differenzen.

In [ ]:
# TODO 5
rng = np.random.default_rng(SEED)
d_rau = None
p_rau = None
p_rau_exakt = None

zeige(None if d_rau is None else d_rau.value_counts().sort_index().to_dict(), p_rau, p_rau_exakt)

In [ ]:
check("TODO 5 Differenzen je Schicht", lambda: len(d_rau) == 60 and d_rau.value_counts().sort_index().to_dict() == {-1: 6, 0: 51, 1: 3} and abs(d_rau.mean() + 0.05) < 1e-9)
check("TODO 5 p-Wert im Block", lambda: abs(p_rau - 0.508) < 0.025 and abs(p_rau_exakt - 0.5078) < 0.0005)

Mit A waren 14 Lose rau, mit B 11. In 51 Schichten waren beide Lose gleich, in 6 war nur das Los mit A rau, in 3 nur das mit B. Der Permutationstest liefert $p = 0{,}514$, exakt sind es 0,508. Für die Rauheit selbst lag der p-Wert im Block bei 0,078. Wer eine Messung an einer Schwelle in ja und nein teilt, wirft weg, wie weit ein Los darunter oder darüber liegt: Nur 9 der 60 Schichten tragen noch zum Test bei.

## 4 · Genauer messen mit Blöcken und Kovariaten
*Lerneinheit 4 · Skript Abschnitt 4*

Die gepaarte Auswertung lässt sich als Regression mit einer Dummy-Variable je Schicht schreiben. Weitere Merkmale kommen als zusätzliche Spalten dazu, etwa die Härte als **Kovariate**. Das ist eine Kovarianzanalyse. Jede Spalte, die einen Teil der Streuung innerhalb der Schichten erklärt, macht den Standardfehler der Scheibe kleiner. Erlaubt sind nur Merkmale, die vor der Behandlung feststehen.

Die nächste Zelle hängt die Härte wie im Skript an `versuch` und passt `m_block` und `m_haerte` an.

In [ ]:
# Die Härte kommt wie im Skript an versuch. Die Zelle lässt sich mehrmals ausführen.
if "haerte_hrc" not in versuch.columns:
    versuch = versuch.merge(haerterei[["charge_id", "haerte_hrc"]], on="charge_id", how="left", validate="many_to_one")
m_block = smf.ols("ra_um ~ C(scheibe) + C(block)", data=versuch).fit()
m_haerte = smf.ols("ra_um ~ C(scheibe) + C(block) + haerte_hrc", data=versuch).fit()
for m in [m_block, m_haerte]:
    print(round(m.params["C(scheibe)[T.B]"], 4), round(m.bse["C(scheibe)[T.B]"], 4), round(m.pvalues["C(scheibe)[T.B]"], 3))

### TODO 6 · Reihenfolge als zweiter Faktor
Zwei Lose laufen nacheinander auf derselben Maschine. Erwärmt das erste die Spindel, hängt das zweite davon ab, welche Scheibe vorher lief. Der Münzwurf hat die Reihenfolge zufällig gewählt, nimm sie trotzdem als zweiten Faktor auf.
1. `m_reihe`: die Regression `ra_um ~ C(scheibe) + C(block) + haerte_hrc + C(reihenfolge)` auf `versuch`.
2. `scheibe_b`: ein Wörterbuch mit `"effekt"`, `"se"`, `"p"`, `"unten"` und `"oben"` (95-%-Intervall) für `C(scheibe)[T.B]`. `zweites`: ein Wörterbuch mit `"effekt"` und `"p"` für `C(reihenfolge)[T.2]`, das zweite Los einer Schicht. Fehlt der Koeffizient im Modell, setze `zweites = None`.
3. `b_zweites`: in wie vielen Schichten B als zweites Los lief.

In [ ]:
# TODO 6
m_reihe = None
scheibe_b = None
zweites = None
b_zweites = None

zeige(scheibe_b, zweites, None if m_reihe is None else m_reihe.params["haerte_hrc"], b_zweites)

In [ ]:
check("TODO 6 Effekt der Scheibe", lambda: abs(scheibe_b["effekt"] + 0.0121) < 0.00005 and abs(scheibe_b["se"] - 0.0037) < 0.00005 and abs(scheibe_b["p"] - 0.002) < 0.0005
      and abs(scheibe_b["unten"] + 0.0196) < 0.00005 and abs(scheibe_b["oben"] + 0.0046) < 0.00005 and int(m_reihe.df_resid) == 57)
check("TODO 6 zweites Los", lambda: abs(zweites["effekt"] - 0.0046) < 0.00005 and abs(zweites["p"] - 0.229) < 0.0005 and abs(m_reihe.params["haerte_hrc"] - 0.0125) < 0.00005
      and b_zweites == 37)

Mit der Reihenfolge schätzt die Regression $-0{,}0121$ µm für B, mit Standardfehler 0,0037 und $p = 0{,}002$. Das 95-%-Intervall reicht von $-0{,}0196$ bis $-0{,}0046$ µm. Das zweite Los einer Schicht ist im Mittel 0,0046 µm rauer, mit $p = 0{,}229$ nicht deutlich. B lief in 37 von 60 Schichten als zweites Los und trug den möglichen Nachteil des zweiten Platzes öfter. Rechnet die Regression ihn heraus, liegt B etwas weiter vorn als ohne Reihenfolge mit $-0{,}0111$ µm. Die Härte zählt mit 0,0125 µm je HRC fast genau wie im Skript.

## 5 · Der A/B-Test im Portal
*Lerneinheit 5 · Skript Abschnitt 5*

Im Portal wurde je Konto zugeteilt, also wird je Konto ausgewertet. Die Bestellquote einer Gruppe ist ein Verhältnisschätzer $\hat r = \sum_i x_i / \sum_i n_i$ über die Konten, ihr Standardfehler kommt aus der Delta-Methode oder aus einem Bootstrap, der ganze Konten zieht. Der **Designeffekt** ist das Verhältnis der Varianz über Konten zur Varianz, die unabhängige Sitzungen hätten. Mit ihm lassen sich Formeln für unabhängige Einheiten auf Konten umrechnen.

Die nächste Zelle lädt die Sitzungen und baut `je_konto`, `quote_mit_se`, `vergleich` und `gesamt` wie im Skript. `se_naiv` ist der Standardfehler für unabhängige Sitzungen, `DEFF` der Designeffekt.

In [ ]:
sitzungen = pd.read_csv("daten/portal_sitzungen.csv", parse_dates=["datum"])
quote = sitzungen.groupby("gruppe")["bestellt"].agg(["mean", "size"])
p_a, p_b = quote["mean"]
se_naiv = np.sqrt((quote["mean"] * (1 - quote["mean"]) / quote["size"]).sum())
je_konto = (sitzungen.groupby(["konto", "gruppe"])
            .agg(sitzungen=("bestellt", "size"), bestellungen=("bestellt", "sum"), umsatz=("umsatz_eur", "sum")).reset_index())


def quote_mit_se(k):
    r = k["bestellungen"].sum() / k["sitzungen"].sum()
    rest = k["bestellungen"] - r * k["sitzungen"]
    return r, np.sqrt((rest ** 2).sum() / (len(k) - 1) / len(k)) / k["sitzungen"].mean()


def vergleich(k):
    (r_a, se_a), (r_b, se_b) = [quote_mit_se(k[k["gruppe"] == g]) for g in "AB"]
    diff, se = r_b - r_a, np.hypot(se_a, se_b)
    return pd.Series({"A": r_a, "B": r_b, "diff": diff, "se": se, "z": diff / se, "p": 2 * stats.norm.sf(abs(diff / se))})


gesamt = vergleich(je_konto)
DEFF = (gesamt["se"] / se_naiv) ** 2
print(gesamt.round(4).to_dict(), round(se_naiv, 4), round(DEFF, 2))

### TODO 7 · Teststärke und Laufzeit des Portaltests
Hätte der Test einen kleineren Anstieg gefunden? Plane rückwärts mit der Formel für zwei Anteile aus dem Skript.
1. `z_a`, `z_b`: die Quantile der Normalverteilung für $\alpha = 5\,\%$ zweiseitig und 80 % Teststärke, wie im Skript.
2. `n_sitzungen`: die Sitzungen je Gruppe für einen relativen Anstieg um 10 % gegenüber der Quote von A, also $p_B = 1{,}1\,p_A$ mit $p_A$ aus `gesamt["A"]`, so gerechnet, als wären die Sitzungen unabhängig. `n_sitzungen_deff`: dasselbe mal `DEFF`. `n_konten`: in Konten umgerechnet, mit den mittleren Sitzungen je Konto im Versuch (`len(sitzungen) / len(konten)`).
3. `staerke`: die Teststärke des durchgeführten Tests für diesen Anstieg, $\Phi\big(\delta/\mathrm{SE} - z_{1-\alpha/2}\big)$ mit dem Standardfehler über Konten aus `gesamt["se"]`.

In [ ]:
# TODO 7
z_a, z_b = None, None
n_sitzungen = None
n_sitzungen_deff = None
n_konten = None
staerke = None

zeige(n_sitzungen, n_sitzungen_deff, n_konten, staerke)

In [ ]:
check("TODO 7 Sitzungen und Konten", lambda: abs(n_sitzungen - 11084) < 1 and abs(n_sitzungen_deff - 19727) < 1.5 and abs(n_konten - 2847) < 1)
check("TODO 7 Teststärke", lambda: abs(z_a - 1.96) < 0.001 and abs(z_b - 0.8416) < 0.0001 and abs(staerke - 0.456) < 0.0005)

In [ ]:
# Vorgabe: Ist TODO 7 noch offen, legt diese Zelle z_a und z_b an. In der Lösung ändert sie nichts.
if z_a is None or z_b is None:
    z_a, z_b = stats.norm.ppf(0.975), stats.norm.ppf(0.80)
print(round(z_a, 4), round(z_b, 4))

Als unabhängige Sitzungen bräuchte der Test 11.084 Sitzungen je Gruppe. Der Designeffekt von 1,78 macht daraus 19.727 Sitzungen oder 2.847 Konten je Gruppe, mehr als doppelt so viele, wie der Test in jeder Gruppe hatte. Für einen Anstieg um 10 % hatte er nur eine Teststärke von 45,6 %. Gemessen hat er 17,6 %, einen Anstieg, den er mit größerer Wahrscheinlichkeit findet. Wer mit Sitzungen plant, obwohl Konten zugeteilt werden, plant einen zu kleinen Test.

### TODO 8 · Zuteilung im Portal nach Woche und Branche
Im Skript passt das Verhältnis der Gruppen insgesamt zu 1/2. Ein Fehler könnte aber nur ab einem bestimmten Tag oder nur bei einer Branche auftreten.
1. `woche`: für jedes Konto die Versuchswoche des ersten Besuchs als Zahl von 1 bis 4. Die erste Versuchswoche beginnt am Montag, dem 31. August 2026.
2. `je_woche`: eine Tabelle mit einer Zeile je Versuchswoche (Index 1 bis 4) und den Spalten `konten`, `B` und `p`, dem p-Wert von `stats.binomtest` gegen 1/2.
3. `p_branche`: der p-Wert von `stats.chi2_contingency` für die Kreuztabelle von Branche (Zeilen) und Gruppe (Spalten).

In [ ]:
# TODO 8
woche = None
je_woche = None
p_branche = None

print(None if je_woche is None else je_woche.round(3))
zeige(p_branche)

In [ ]:
check("TODO 8 je Woche", lambda: list(je_woche.index) == [1, 2, 3, 4] and je_woche["konten"].astype(int).tolist() == [1746, 451, 225, 126]
      and je_woche["B"].astype(int).tolist() == [892, 230, 120, 67] and np.allclose(je_woche["p"], [0.376, 0.706, 0.351, 0.533], atol=0.0005))
check("TODO 8 Branchen", lambda: abs(p_branche - 0.227) < 0.0005)

In der ersten Versuchswoche besuchten 1.746 Konten das Portal zum ersten Mal, 892 davon kamen nach B ($p = 0{,}376$). Danach werden es weniger: 451, 225 und 126 neue Konten, davon 230, 120 und 67 in B. Kein p-Wert liegt unter 0,35, und die Branchen verteilen sich mit $p = 0{,}227$ unauffällig auf A und B. Ein Fehler, der nur ab einem bestimmten Tag oder nur eine Branche träfe, würde sich hier zeigen. Dass die erste Versuchswoche die meisten neuen Konten hat, ist zu erwarten: Wer regelmäßig ins Portal schaut, kommt meist schon in den ersten Tagen.

### TODO 9 · Bootstrap über Konten
Die Delta-Methode ist eine Näherung. Der Bootstrap aus Modul 5 kommt ohne Formel aus, wenn er ganze Konten zieht, wie zugeteilt wurde.
1. `boot`: ein Wörterbuch mit den Schlüsseln „A“ und „B“ und je 2.000 Bestellquoten. Lege `rng = np.random.default_rng(SEED)` an. Für jede Gruppe, erst A, dann B, nimmst du die Konten in der Reihenfolge von `je_konto` und ziehst mit `rng.integers(0, m, size=(2000, m))` alle Stichproben auf einmal; $m$ ist die Zahl der Konten der Gruppe. Die Quote einer Stichprobe ist die Summe der Bestellungen durch die Summe der Sitzungen der gezogenen Konten.
2. `se_boot`: die Standardabweichung (`ddof=1`) der 2.000 Differenzen B minus A. `ki_boot`: das 2,5-%- und das 97,5-%-Quantil der Differenzen. `ki_boot_rel`: dieselben Quantile für den relativen Anstieg `boot["B"] / boot["A"] - 1`.

In [ ]:
# TODO 9
rng = np.random.default_rng(SEED)
boot = None
se_boot = None
ki_boot = None
ki_boot_rel = None

zeige(se_boot, ki_boot, ki_boot_rel)

In [ ]:
check("TODO 9 Standardfehler über Konten", lambda: len(boot["A"]) == 2000 and len(boot["B"]) == 2000 and abs(se_boot - gesamt["se"]) < 0.0006)
check("TODO 9 Intervalle", lambda: abs(ki_boot[0] - 0.0088) < 0.0022 and abs(ki_boot[1] - 0.0364) < 0.0022
      and abs(ki_boot_rel[0] - 0.064) < 0.018 and abs(ki_boot_rel[1] - 0.300) < 0.02)

Der Bootstrap über Konten liefert einen Standardfehler von 0,0070, so viel wie die Delta-Methode. Das Perzentilintervall der Differenz reicht von 0,88 bis 3,64 Prozentpunkte, fast dasselbe wie mit der Delta-Methode, 0,90 bis 3,62. Der relative Anstieg liegt zwischen 6,4 und 30,0 %. Dieses Intervall nimmt die Unsicherheit der Quote in A mit und ist deshalb etwas breiter als 7 bis 28,2 % im Skript.

Die nächste Zelle rechnet CUPED mit dem Umsatz der Vorperiode wie im Skript. `je_konto` bekommt dabei Branche, Vorjahresumsatz, Umsatz der Vorperiode und die Spalte `umsatz_cuped`.

In [ ]:
je_konto = je_konto.merge(konten[["konto", "branche", "umsatz_vorjahr_eur", "umsatz_vorperiode_eur"]], on="konto")
x_vp = je_konto["umsatz_vorperiode_eur"]
theta = np.cov(x_vp, je_konto["umsatz"])[0, 1] / x_vp.var()
je_konto["umsatz_cuped"] = je_konto["umsatz"] - theta * (x_vp - x_vp.mean())
c_a, c_b = [je_konto.loc[je_konto["gruppe"] == g, "umsatz_cuped"] for g in "AB"]
d_c, se_c = c_b.mean() - c_a.mean(), np.sqrt(c_a.var() / len(c_a) + c_b.var() / len(c_b))
print(je_konto.shape, round(theta, 3), round(d_c, 1), round(se_c, 1), round(2 * stats.norm.sf(abs(d_c / se_c)), 3))

### TODO 10 · CUPED mit dem Vorjahresumsatz
Im Skript senkt der Umsatz der Vorperiode die Varianz um 23,2 %. Rechne CUPED mit dem Vorjahresumsatz und sieh nach, was mit der Schätzung passiert. Lass `je_konto` dabei unverändert.
1. `cuped_vj`: ein Wörterbuch mit `"r"` (Korrelation von `umsatz_vorjahr_eur` und `umsatz`), `"theta"`, `"weniger_varianz"` (Anteil, um den die Varianz sinkt), `"diff"` (B minus A nach CUPED), `"se"` (wie im Skript aus den Varianzen je Gruppe) und `"p"` (zweiseitig über die Normalverteilung).
2. `vorjahr_mittel`: der mittlere Vorjahresumsatz je Gruppe als Series.
3. `mit_beiden`: ein Wörterbuch mit `"diff"`, `"se"` und `"p"` für `C(gruppe)[T.B]` aus `smf.ols("umsatz ~ C(gruppe) + umsatz_vorperiode_eur + umsatz_vorjahr_eur", data=je_konto)`.

In [ ]:
# TODO 10
cuped_vj = None
vorjahr_mittel = None
mit_beiden = None

zeige(cuped_vj, None if vorjahr_mittel is None else vorjahr_mittel.to_dict(), mit_beiden)

In [ ]:
check("TODO 10 CUPED mit dem Vorjahr", lambda: abs(cuped_vj["r"] - 0.275) < 0.0005 and abs(cuped_vj["theta"] - 0.0326) < 0.00005
      and abs(cuped_vj["weniger_varianz"] - 0.076) < 0.0005 and abs(cuped_vj["diff"] - 173.1) < 0.05 and abs(cuped_vj["se"] - 89.7) < 0.05 and abs(cuped_vj["p"] - 0.054) < 0.0005
      and abs(vorjahr_mittel["A"] - 14747.7) < 0.05 and abs(vorjahr_mittel["B"] - 13514.8) < 0.05)
check("TODO 10 beide Kovariaten", lambda: abs(mit_beiden["diff"] - 132.6) < 0.05 and abs(mit_beiden["se"] - 80.7) < 0.05 and abs(mit_beiden["p"] - 0.1) < 0.005
      and "umsatz_cuped" in je_konto.columns and je_konto.shape == (2548, 9))

Der Vorjahresumsatz hängt nur mit 0,275 am Umsatz im Versuch, $\theta = 0{,}0326$, und die Varianz sinkt um 7,6 %. Die Schätzung steigt aber auf 173,1 € mit Standardfehler 89,7 und $p = 0{,}054$. Der Grund: A hatte im Vorjahr im Mittel mehr Umsatz, 14.747,7 gegen 13.514,8 €, und CUPED rechnet diesen zufälligen Vorsprung heraus. Mit beiden Kovariaten schätzt die Regression 132,6 € mit Standardfehler 80,7 und $p = 0{,}1$. Eine Kovariate senkt den Standardfehler so weit, wie sie den Umsatz erklärt, und verschiebt die Schätzung, wenn sich die Gruppen in ihr zufällig unterscheiden. Welche Kovariate zählt, gehört deshalb vor dem Versuch in den Plan.

## 6 · Fallstricke: Peeking, Teilgruppen, Klumpen
*Lerneinheit 6 · Skript Abschnitt 6*

Wer während des Tests jeden Tag nachsieht und beim ersten $|z| > 1{,}96$ aufhört, hat nicht die Fehlerrate 5 %. Eine **A/A-Simulation** zeigt das: Konten, die alle dieselbe Seite sahen, werden zufällig in zwei Hälften geteilt, und jeder Unterschied ist Zufall. Bei Teilgruppen gilt dasselbe wie bei vielen Tests: Je mehr Gruppen, desto eher fällt eine zufällig auf.

Die nächste Zelle rechnet wie im Skript das kumulierte $z$ für jeden der 28 Versuchstage (`tage`, `verlauf`) und zeigt den Verlauf.

In [ ]:
tage = np.sort(sitzungen["datum"].unique())
verlauf = pd.DataFrame([vergleich(sitzungen[sitzungen["datum"] <= tag].groupby(["konto", "gruppe"])
                                  .agg(sitzungen=("bestellt", "size"), bestellungen=("bestellt", "sum")).reset_index()) for tag in tage],
                       index=pd.DatetimeIndex(tage))
print(len(tage), verlauf["z"].round(2).iloc[[0, 2, 27]].tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(verlauf.index, verlauf["z"], marker="o", ms=3, color="tab:blue")
ax.axhline(1.96, color="gray", ls="--", lw=1)
ax.set_ylabel("kumuliertes z")
ax.set_title("Bestellquote über Konten, ausgewertet mit allen Sitzungen bis zum Tag")
fig.autofmt_xdate()
plt.show()

### TODO 11 · A/A-Simulation zum Peeking
Die Konten aus A sahen alle die bisherige Seite. Teile sie 400-mal zufällig in zwei Hälften und werte jede Teilung an jedem Tag so aus wie den echten Test. Eine Schleife über 400 Teilungen und 28 Tage mit `vergleich` dauert im Browser zu lange, rechne deshalb mit Matrizen.
1. `X` und `N`: für jedes Konto aus A (Zeilen, nach Kennung sortiert) und jeden Tag aus `tage` (Spalten) die Bestellungen und Sitzungen bis einschließlich dieses Tags, als Arrays der Form (1.239, 28). Hilfreich: `pivot_table` mit `aggfunc="sum"` und `aggfunc="size"`, dann `reindex(columns=tage, fill_value=0)` und `cumsum(axis=1)`.
2. `haelfte`: ein boolesches Array der Form (400, 1.239) aus `rng = np.random.default_rng(SEED)` und `rng.random((400, m)) < 0.5`. True heißt: Das Konto liegt in dieser Teilung in der ersten Hälfte.
3. `z_aa`: das kumulierte $z$ über Konten (erste Hälfte minus zweite) für jede Teilung und jeden Tag, Form (400, 28). Ein Konto zählt an einem Tag mit, sobald es eine Sitzung hatte. Mit einer 0/1-Matrix $H$ der Hälften ergeben Produkte wie `H @ X` die Summen je Teilung und Tag, und die Quadratsumme der Reste ist $\sum x^2 - 2r\sum xn + r^2\sum n^2$.
4. `treffer`: ein Wörterbuch, wie viele der 400 Teilungen $|z| > 1{,}96$ zeigen, mit den Schlüsseln `"Ende"` (nur am letzten Tag), `"täglich"` (an irgendeinem Tag) und `"wöchentlich"` (an einem der vier Sonntage, den Tagen mit Index 6, 13, 20 und 27).

In [ ]:
# TODO 11
rng = np.random.default_rng(SEED)
X, N = None, None
haelfte = None
z_aa = None
treffer = None

zeige(None if z_aa is None else np.shape(z_aa), treffer)

In [ ]:
def _eine_teilung(i):
    k = je_konto[je_konto["gruppe"] == "A"]
    return vergleich(k.assign(gruppe=np.where(haelfte[i], "B", "A")))["z"]

check("TODO 11 z wie mit vergleich", lambda: np.shape(X) == (1239, 28) and np.shape(haelfte) == (400, 1239) and np.shape(z_aa) == (400, 28)
      and abs(_eine_teilung(0) - z_aa[0, -1]) < 1e-9 and abs(_eine_teilung(399) - z_aa[399, -1]) < 1e-9)
check("TODO 11 Treffer", lambda: treffer["Ende"] == int((np.abs(z_aa[:, -1]) > 1.96).sum()) and treffer["täglich"] == int((np.abs(z_aa) > 1.96).any(axis=1).sum())
      and abs(treffer["Ende"] - 20) <= 18 and abs(treffer["täglich"] - 93) <= 34 and abs(treffer["wöchentlich"] - 46) <= 26
      and treffer["Ende"] <= treffer["wöchentlich"] <= treffer["täglich"])

In [ ]:
# Grafik zu TODO 11: zeigt die ersten 60 Teilungen, sobald z_aa berechnet ist.
fig, ax = plt.subplots(figsize=(8, 3.2))
if z_aa is not None:
    ax.plot(np.arange(1, 29), np.asarray(z_aa)[:60].T, color="tab:blue", lw=0.6, alpha=0.5)
ax.axhline(1.96, color="gray", ls="--", lw=1)
ax.axhline(-1.96, color="gray", ls="--", lw=1)
ax.set_xlabel("Versuchstag")
ax.set_ylabel("kumuliertes z")
ax.set_title("A/A-Teilungen: jeder Unterschied ist Zufall")
plt.show()

Wer nur am Ende nachsieht, findet in 20 von 400 Teilungen einen Unterschied, 5 %. Wer täglich nachsieht und beim ersten Überschreiten aufhört, findet ihn in 93 Teilungen, 23 %, wer wöchentlich am Sonntag nachsieht, in 46, 11,5 %. Die erste und die letzte Teilung geben am letzten Tag dasselbe $z$ wie `vergleich`, die Matrizen rechnen also dieselbe Auswertung für alle Teilungen zugleich.

Die nächste Zelle wertet wie im Skript die sechs Branchen aus und bereinigt die p-Werte nach Holm (`teil`).

In [ ]:
teil = je_konto.groupby("branche")[["gruppe", "sitzungen", "bestellungen"]].apply(vergleich)
teil["holm"] = multipletests(teil["p"], method="holm")[1]
print(teil[["A", "B", "diff", "p", "holm"]].round(4))

### TODO 12 · Teilgruppe mit Begründung
Der Knopf zeigt die zuletzt bestellten Teile. Konten, die in den vier Wochen vor dem Versuch bestellt haben, sehen dort also etwas, Konten ohne Bestellung kaum. Diese Teilgruppe ist vorher begründet. Prüfe sie.
1. `mit_vp`: die Auswertung mit `vergleich` getrennt für Konten ohne und mit Umsatz in der Vorperiode, als Tabelle mit dem Index `False` und `True`. Gruppiere `je_konto` nach `je_konto["umsatz_vorperiode_eur"] > 0`.
2. `holm_12`: die nach Holm bereinigten p-Werte der beiden Tests als Series mit demselben Index.
3. `unterschied`: ein Wörterbuch mit `"diff"` (Differenz mit Vorperiode minus Differenz ohne), `"se"` (aus beiden Standardfehlern wie bei zwei unabhängigen Schätzungen) und `"p"` (zweiseitig). `unterschied_belegt`: True, wenn der Unterschied zwischen den Teilgruppen auf dem Niveau 5 % nachgewiesen ist.

In [ ]:
# TODO 12
mit_vp = None
holm_12 = None
unterschied = None
unterschied_belegt = None

print(None if mit_vp is None else mit_vp.round(4))
zeige(None if holm_12 is None else holm_12.to_dict(), unterschied, unterschied_belegt)

In [ ]:
check("TODO 12 Teilgruppen mit Holm", lambda: list(mit_vp.index) == [False, True] and abs(mit_vp.loc[True, "diff"] - 0.0267) < 0.00005 and abs(mit_vp.loc[True, "p"] - 0.0042) < 0.00005
      and abs(mit_vp.loc[False, "diff"] - 0.0117) < 0.00005 and abs(mit_vp.loc[False, "p"] - 0.184) < 0.0005 and abs(holm_12[True] - 0.0084) < 0.00005 and abs(holm_12[False] - 0.184) < 0.0005)
check("TODO 12 Unterschied der Differenzen", lambda: abs(unterschied["diff"] - 0.0150) < 0.00005 and abs(unterschied["se"] - 0.0128) < 0.00005 and abs(unterschied["p"] - 0.243) < 0.0005
      and unterschied_belegt is False)

Bei den 1.130 Konten mit Umsatz in der Vorperiode steigt die Quote um 2,67 Prozentpunkte ($p = 0{,}0042$, nach Holm 0,0084), bei den 1.418 Konten ohne um 1,17 Prozentpunkte ($p = 0{,}184$). Der Unterschied der beiden Differenzen beträgt 1,50 Prozentpunkte, mit Standardfehler 1,28 und $p = 0{,}243$. Dass ein Test unter 0,05 liegt und der andere nicht, zeigt keinen Unterschied zwischen den Teilgruppen. Nachgewiesen ist ein Anstieg bei den Konten mit Vorperiode, auch nach Holm; ob der Knopf dort stärker wirkt als bei den übrigen, lässt der Versuch offen.

## 7 · Entscheiden
*Lerneinheit 7 · Skript Abschnitt 7*

Ein Ergebnisbericht nennt für jede Kennzahl beide Gruppen, den Unterschied, das Intervall und den p-Wert, dazu die Einheit. Bei Anteilen heißt der Unterschied in Prozentpunkten, bei einem relativen Anstieg in Prozent. Entschieden wird mit der Regel aus dem Plan: Die Hauptkennzahl muss steigen, die Leitplanke darf nicht deutlich sinken.

### TODO 13 · Entscheidungsbericht als Tabelle
1. `bericht`: eine Tabelle mit den Zeilen „Bestellquote je Sitzung“, „Konten mit Bestellung“, „Bestellwert je Bestellung“ und „Umsatz je Konto mit CUPED“ (in dieser Reihenfolge) und den Spalten `A`, `B`, `Unterschied`, `unten`, `oben` und `p`. Quoten und Anteile stehen in Prozent, ihre Unterschiede und Intervalle in Prozentpunkten, Beträge in Euro.
   - Bestellquote je Sitzung: aus `gesamt`, Intervall $\pm 1{,}96\,\mathrm{SE}$.
   - Konten mit Bestellung: Anteil der Konten mit mindestens einer Bestellung, Standardfehler für zwei Anteile, Intervall und p-Wert über die Normalverteilung.
   - Bestellwert je Bestellung: Welch-Test über die Sitzungen mit Bestellung, mit seinem Intervall.
   - Umsatz je Konto: `A` und `B` wie im Skript ohne Korrektur, Unterschied, Intervall und p-Wert mit CUPED aus `d_c` und `se_c`.
2. `empfehlung`: zwei Sätze als Text. Der erste nennt den Anstieg der Hauptkennzahl mit Einheit und Intervall, der zweite die Empfehlung. Zahlen mit Dezimalkomma.

In [ ]:
# TODO 13
bericht = None
empfehlung = None

print(None if bericht is None else bericht.round(4).to_string())
print(empfehlung)

In [ ]:
check("TODO 13 Tabelle", lambda: list(bericht.index) == ["Bestellquote je Sitzung", "Konten mit Bestellung", "Bestellwert je Bestellung", "Umsatz je Konto mit CUPED"]
      and list(bericht.columns) == ["A", "B", "Unterschied", "unten", "oben", "p"]
      and np.allclose(bericht.iloc[0], [12.86, 15.12, 2.26, 0.90, 3.62, 0.0012], atol=0.005) and np.allclose(bericht.iloc[1], [43.99, 47.82, 3.84, -0.03, 7.70, 0.052], atol=0.005)
      and np.allclose(bericht.iloc[2], [1132.9, 1086.5, -46.4, -158.0, 65.2, 0.415], atol=0.05) and np.allclose(bericht.iloc[3], [1007.6, 1140.4, 106.0, -54.3, 266.3, 0.195], atol=0.05))
check("TODO 13 Empfehlung", lambda: isinstance(empfehlung, str) and "2,26 Prozentpunkte" in empfehlung and "2,26 %" not in empfehlung and "0,90" in empfehlung and "3,62" in empfehlung
      and empfehlung.rstrip().endswith(".") and empfehlung.count(". ") == 1)

Die Bestellquote je Sitzung steigt von 12,86 auf 15,12 %, um 2,26 Prozentpunkte (95-%-Intervall 0,90 bis 3,62, $p = 0{,}0012$). Mindestens eine Bestellung haben in A 43,99 % der Konten, in B 47,82 %, ein Plus von 3,84 Prozentpunkten mit Intervall von $-0{,}03$ bis 7,70 und $p = 0{,}052$. Der Bestellwert sinkt um 46,4 € je Bestellung, das Intervall reicht von $-158{,}0$ bis 65,2 € und $p = 0{,}415$; er ist nur beschreibend, weil die Variante selbst mitbestimmt, welche Sitzungen zu einer Bestellung werden. Der Umsatz je Konto liegt mit CUPED um 106,0 € höher, mit Intervall von $-54{,}3$ bis 266,3 € und $p = 0{,}195$. Die Hauptkennzahl steigt, die Leitplanke hält. Wer „+2,26 %“ schreibt, meint einen relativen Anstieg, der fast achtmal kleiner ist als die gemessenen 17,6 %.

### Bonus 1 · Neuheitseffekt je Woche
Zieht der Knopf nur anfangs Aufmerksamkeit auf sich? Werte jede Versuchswoche für sich aus.
1. `je_versuchswoche`: eine Tabelle mit einer Zeile je Versuchswoche (Index 1 bis 4) und den Spalten `relativ`, `unten` und `oben`: der relative Anstieg der Bestellquote in Prozent der Quote von A und die Grenzen von $\text{diff} \pm 1{,}96\,\mathrm{SE}$, ebenfalls geteilt durch die Quote von A. Nimm für jede Woche nur ihre Sitzungen, bilde daraus die Tabelle je Konto wie für `verlauf` und werte sie mit `vergleich` aus.

In [ ]:
# Bonus 1
je_versuchswoche = None

print(None if je_versuchswoche is None else je_versuchswoche.round(1))

In [ ]:
check("Bonus 1 Neuheitseffekt", lambda: list(je_versuchswoche.index) == [1, 2, 3, 4] and list(je_versuchswoche.columns) == ["relativ", "unten", "oben"]
      and np.allclose(je_versuchswoche["relativ"], [25.3, 14.9, 24.5, 6.5], atol=0.05) and np.allclose(je_versuchswoche["unten"], [8.1, -2.4, 6.4, -10.3], atol=0.05)
      and np.allclose(je_versuchswoche["oben"], [42.4, 32.3, 42.7, 23.3], atol=0.05))

Je Versuchswoche liegt der relative Anstieg bei 25,3 %, 14,9 %, 24,5 % und 6,5 %. Die Intervalle reichen von 8,1 bis 42,4 %, von $-2{,}4$ bis 32,3 %, von 6,4 bis 42,7 % und von $-10{,}3$ bis 23,3 %. Die vierte Versuchswoche liegt am niedrigsten, aber alle Intervalle überlappen sich. Ein Rückgang über die Zeit ist mit vier Wochen nicht nachgewiesen. Wer einen Neuheitseffekt ausschließen will, lässt den Test länger laufen oder vergleicht neue und wiederkehrende Konten.

### Bonus 2 · Teststärke für den Umsatz mit CUPED
Der Umsatz je Konto ist im Test nicht nachweisbar gestiegen. Wie groß hätte der Test sein müssen?
1. `konten_100`: ein Wörterbuch mit den Schlüsseln `"ohne CUPED"` und `"mit CUPED"` und den Konten je Gruppe für 80 % Teststärke bei 100 € mehr Umsatz je Konto, mit $n \approx (z_{1-\alpha/2} + z_{1-\beta})^2 (s_A^2 + s_B^2)/\delta^2$ aus den Varianzen je Gruppe von `umsatz` beziehungsweise `umsatz_cuped`.
2. `mde`: ein Wörterbuch mit denselben Schlüsseln und dem kleinsten Unterschied, den der durchgeführte Test mit 80 % Teststärke findet, $(z_{1-\alpha/2} + z_{1-\beta})\sqrt{s_A^2/m_A + s_B^2/m_B}$ mit den Konten $m_A$ und $m_B$ je Gruppe.

In [ ]:
# Bonus 2
konten_100 = None
mde = None

zeige(konten_100, mde)

In [ ]:
check("Bonus 2 Teststärke für den Umsatz", lambda: abs(konten_100["ohne CUPED"] - 8715) < 1 and abs(konten_100["mit CUPED"] - 6701) < 1
      and abs(mde["ohne CUPED"] - 261.2) < 0.05 and abs(mde["mit CUPED"] - 229.1) < 0.05)

Für 100 € mehr Umsatz je Konto bräuchte der Test ohne CUPED 8.715 Konten je Gruppe, mit CUPED 6.701, gut ein Fünftel weniger. Mit den vorhandenen Konten findet er mit 80 % Teststärke erst einen Unterschied von 261,2 € ohne und 229,1 € mit CUPED. Beim Umsatz ist der Test also zu klein für Unterschiede, die sich lohnen. Die Leitplanke prüft er nur grob: Einen großen Rückgang schließt er aus, einen kleinen nicht.

## Zum Nachdenken

1. In TODO 1 verschlechtern sich die besten Maschinen ohne jede Maßnahme. Wie würdest du eine Belobigung der besten Schichten so planen, dass sich ihre Wirkung zeigen lässt?
2. In TODO 4 liegt die obere Grenze der Schleifzeit genau an den erlaubten 3 %. Würdest du B einführen? Was müsste im Plan stehen, damit die Antwort vorher feststeht?
3. Der Vorjahresumsatz verschiebt in TODO 10 die Schätzung von 106,0 auf 173,1 €. Warum darfst du nach dem Versuch nicht die Kovariate wählen, die dir am besten gefällt?
4. Die A/A-Simulation in TODO 11 zeigt bei täglichem Nachsehen 23 % falsche Treffer. Das Portal möchte trotzdem jeden Tag auf die Zahlen schauen. Was schlägst du vor?
5. In TODO 12 ist der Anstieg bei Konten mit Vorperiode nachgewiesen, bei den übrigen nicht. Darf das Portal den Knopf nur diesen Konten zeigen?

<details><summary>Hinweise zu den Antworten</summary>

1. Mit einer Vergleichsgruppe, die genauso ausgewählt wird: Unter den besten Schichten entscheidet der Zufall, welche gelobt werden. Verglichen werden gelobte und nicht gelobte Schichten im Folgemonat. Beide fallen zur Mitte zurück, der Unterschied zwischen ihnen zeigt die Wirkung.
2. Die Regel war „höchstens 3 %“, und die Schätzung liegt mit 2,55 % darunter. Ob die Schätzung oder die obere Grenze zählt, sollte im Plan stehen. Die Kosten der längeren Schleifzeit gehören mit vorsichtigen Annahmen in die Entscheidung, wie in Abschnitt 7 des Skripts.
3. Jede Kovariate liefert eine etwas andere Schätzung. Wer nach dem Versuch die mit dem kleinsten p-Wert nimmt, wählt unter vielen Auswertungen, wie bei vielen Kennzahlen oder Teilgruppen. Die Kovariate steht vorher im Plan, am besten die, die den Umsatz am stärksten erklärt.
4. Ansehen ist erlaubt, Entscheiden nicht. Laufzeit und Stichprobe stehen vorher fest, ausgewertet wird am Ende. Wer früh aufhören will, nimmt ein sequenzielles Verfahren mit strengeren Grenzen für die Zwischenblicke, und gegen grobe Fehler hilft eine tägliche Prüfung der Zuteilung.
5. Dafür reicht der Versuch nicht. Ein Unterschied zwischen den Teilgruppen ist mit $p = 0{,}243$ nicht nachgewiesen, und auch bei den Konten ohne Vorperiode liegt die Schätzung über null. Wer den Knopf nur einem Teil zeigen will, braucht dafür einen eigenen Grund oder einen eigenen Test.
</details>

## Wie es weitergeht

Du hast gesehen, wie die Regression zur Mitte auch die Besten trifft, eine Zuteilung reproduzierbar gezogen und geprüft, ob die Kovariate vor der Behandlung feststand. Im Werkversuch hast du die Schleifzeit gepaart ausgewertet, raue Lose mit Vorzeichentausch geprüft und die Reihenfolge als zweiten Faktor aufgenommen. Im Portal hast du die Teststärke geplant, die Zuteilung je Woche und Branche geprüft, einen Bootstrap über Konten, CUPED mit dem Vorjahr und eine A/A-Simulation gerechnet, eine begründete Teilgruppe getestet und einen Entscheidungsbericht geschrieben. Nicht immer lässt sich randomisieren. **Modul 13 „Kausale Inferenz mit Beobachtungsdaten“** fragt: Was tun, wenn man nicht randomisieren kann?